In [1]:
import torch
torch._dynamo.config.cache_size_limit = 64
torch._dynamo.config.suppress_errors = True
torch.set_float32_matmul_precision('high')

import ChatTTS
from IPython.display import Audio
import numpy

c:\Users\HenryLi\.conda\envs\VUI\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\HenryLi\.conda\envs\VUI\lib\site-packages\vocos\spectral_ops.py:2: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 1.21.0)
  import scipy


## Load Models

In [2]:
chat = ChatTTS.Chat()
chat.load_models()

# Use force_redownload=True if the weights updated.
# chat.load_models(force_redownload=True)

# If you download the weights manually, set source='locals'.
# chat.load_models(source='local', local_path='YOUR LOCAL PATH')

INFO:ChatTTS.core:Load from cache: C:\Users\HenryLi/.cache/huggingface\hub/models--2Noise--ChatTTS/snapshots\cc14302f34d7855eb3420d1fd48345012ff1460d
INFO:ChatTTS.core:use cuda:0
INFO:ChatTTS.core:vocos loaded.
INFO:ChatTTS.core:dvae loaded.
INFO:ChatTTS.core:gpt loaded.
INFO:ChatTTS.core:decoder loaded.
INFO:ChatTTS.core:tokenizer loaded.
INFO:ChatTTS.core:All initialized.


## Inference

### Batch infer

In [3]:
texts = ["我觉得像我们这些写程序的人，他，我觉得多多少少可能会对开源有一种情怀在吧我觉得开源是一个很好的形式。现在其实最先进的技术掌握在一些公司的手里的话，就他们并不会轻易的开放给所有的人用。"]*3     
        
wavs = chat.infer(texts)

INFO:ChatTTS.core:All initialized.
2024-06-18 03:32:40,523 WETEXT INFO found existing fst: c:\Users\HenryLi\.conda\envs\VUI\lib\site-packages\tn\zh_tn_tagger.fst
INFO:wetext-zh_normalizer:found existing fst: c:\Users\HenryLi\.conda\envs\VUI\lib\site-packages\tn\zh_tn_tagger.fst
2024-06-18 03:32:40,525 WETEXT INFO                     c:\Users\HenryLi\.conda\envs\VUI\lib\site-packages\tn\zh_tn_verbalizer.fst
INFO:wetext-zh_normalizer:                    c:\Users\HenryLi\.conda\envs\VUI\lib\site-packages\tn\zh_tn_verbalizer.fst
2024-06-18 03:32:40,526 WETEXT INFO skip building fst for zh_normalizer ...
INFO:wetext-zh_normalizer:skip building fst for zh_normalizer ...
  0%|          | 0/384 [00:00<?, ?it/s]W0618 03:32:43.130235 22740 torch\_dynamo\convert_frame.py:824] WON'T CONVERT _update_causal_mask c:\Users\HenryLi\.conda\envs\VUI\lib\site-packages\transformers\models\llama\modeling_llama.py line 1005 
W0618 03:32:43.130235 22740 torch\_dynamo\convert_frame.py:824] due to: 
W0618 03:32

In [4]:
Audio(wavs[0], rate=24_000, autoplay=True)

In [10]:
Audio(wavs[2], rate=24_000, autoplay=True)

### Custom params

In [11]:
params_infer_code = {'prompt':'[speed_5]', 'temperature':.3}
params_refine_text = {'prompt':'[oral_2][laugh_0][break_6]'}

wav = chat.infer('四川美食可多了，有麻辣火锅、宫保鸡丁、麻婆豆腐、担担面、回锅肉、夫妻肺片等，每样都让人垂涎三尺。', \
    params_refine_text=params_refine_text, params_infer_code=params_infer_code)

INFO:ChatTTS.core:All initialized.
 19%|█▉        | 398/2048 [00:11<00:48, 34.37it/s]


In [12]:
Audio(wav[0], rate=24_000, autoplay=True)

### fix random speaker

In [13]:
rand_spk = chat.sample_random_speaker()
params_infer_code = {'spk_emb' : rand_spk, }

wav = chat.infer('四川美食确实以辣闻名，但也有不辣的选择。比如甜水面、赖汤圆、蛋烘糕、叶儿粑等，这些小吃口味温和，甜而不腻，也很受欢迎。', \
    params_refine_text=params_refine_text, params_infer_code=params_infer_code)

INFO:ChatTTS.core:All initialized.
 27%|██▋       | 547/2048 [00:14<00:39, 37.97it/s]


In [14]:
Audio(wav[0], rate=24_000, autoplay=True)

### Two stage control

In [15]:
text = "So we found being competitive and collaborative was a huge way of staying motivated towards our goals, so one person to call when you fall off, one person who gets you back on then one person to actually do the activity with."
chat.infer(text, refine_text_only=True)

INFO:ChatTTS.core:All initialized.
 NeMo-text-processing :: INFO     :: Creating ClassifyFst grammars.
INFO:NeMo-text-processing:Creating ClassifyFst grammars.
 22%|██▏       | 86/384 [00:01<00:06, 45.40it/s]


['so we found [uv_break] being competitive and collaborative [uv_break] was [uv_break] a huge way of staying motivated [uv_break] towards our goals, [uv_break] so one person to call [uv_break] when you fall off, [uv_break] one person who [uv_break] gets you back [uv_break] on then one person to actually do the activity with.']

In [16]:
text = 'so we found being competitive and collaborative [uv_break] was a huge way of staying [uv_break] motivated towards our goals, [uv_break] so [uv_break] one person to call [uv_break] when you fall off, [uv_break] one person who [uv_break] gets you back [uv_break] on then [uv_break] one person [uv_break] to actually do the activity with.'
wav = chat.infer(text, skip_refine_text=True)

INFO:ChatTTS.core:All initialized.
 50%|█████     | 1025/2048 [00:22<00:22, 45.08it/s]


## LLM Call

In [17]:
from ChatTTS.experimental.llm import llm_api

API_KEY = ''
client = llm_api(api_key=API_KEY,
        base_url="https://api.deepseek.com",
        model="deepseek-chat")

ModuleNotFoundError: No module named 'openai'

In [18]:
user_question = '四川有哪些好吃的美食呢?'
text = client.call(user_question, prompt_version = 'deepseek')
print(text)
text = client.call(text, prompt_version = 'deepseek_TN')
print(text)

NameError: name 'client' is not defined

In [14]:
params_infer_code = {'spk_emb' : rand_spk, 'temperature':.3}

wav = chat.infer(text, params_infer_code=params_infer_code)

INFO:ChatTTS.core:All initialized.
 32%|███▏      | 647/2048 [00:04<00:09, 140.27it/s]
